In [1]:
import sys
import os
from pathlib import Path
import subprocess

# Detect Environment
IN_COLAB: bool = "google.colab" in sys.modules
IN_VERTEX_JOBS: bool = "AIP_MODEL_DIR" in os.environ
IS_LOCAL: bool = not IN_COLAB and not IN_VERTEX_JOBS

# Conditional Setup if environment is in Google Colab
if IN_COLAB:
    # from google.colab import auth
    # auth.authenticate_user()
    print("Running in Colab: Authenticated")
    source_folder: Path = Path("multi_label_classifier").resolve()

    if source_folder.exists():
        print("Repository already cloned")
        os.chdir(source_folder)
        print(f"Working from {source_folder}")
    else:
        # Clone the git repo of the project
        # !git clone https://github.com/i-putu-mahendra-wijaya/multi_label_classifier.git
        os.chdir(source_folder)
        print(f"Working from {source_folder}")

In [2]:
result: object = subprocess.run(
    [sys.executable, "config_creator.py"],
    capture_output=True,
    text=True,
)

print(result)

CompletedProcess(args=['/Users/iputumahendrawijaya/BinusTeaching/multi_label_classifier/venv_mlc/bin/python', 'config_creator.py'], returncode=0, stdout='', stderr='')


In [3]:
from trainer.DaoKaggle.CredentialAccessor import CredentialAccessor as KaggleCrAcc
from trainer.DaoKaggle.ImageDownloader import ImageDownloader as KaggleImageDownloader

2026-01-10 10:33:29.128890: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/iputumahendrawijaya/BinusTeaching/multi_label_classifier/venv_mlc/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [4]:
FASHION_PRODUCT_IMAGES_URL: str = "paramaggarwal/fashion-product-images-small"


In [5]:
ROOT_DIRECTORY: Path = Path.cwd()
DATASET_DIRECTORY: Path = ROOT_DIRECTORY / "datasets"
GCP_SERVICE_ACCOUNT_PATH: Path = ROOT_DIRECTORY / "credentials" / "gcp" / "service_account.json"
KAGGLE_JSON_PATH: Path = ROOT_DIRECTORY / "credentials" / "kaggle" / "kaggle.json"

In [6]:
if KAGGLE_JSON_PATH.exists():
    kagglecracc: KaggleCrAcc = KaggleCrAcc(
        kaggle_json_path = KAGGLE_JSON_PATH,
    )

kaggle_image_downloader: KaggleImageDownloader = KaggleImageDownloader(
    kaggle_credential_accessor = kagglecracc if KAGGLE_JSON_PATH.exists() else None,
)

downloaded_image_path: Path = kaggle_image_downloader.download_image(
    image_dataset_url = FASHION_PRODUCT_IMAGES_URL,
    output_dir = DATASET_DIRECTORY,
)

print(downloaded_image_path)

100%|██████████| 565M/565M [01:35<00:00, 6.22MB/s] 

Extracting files...


/Users/iputumahendrawijaya/BinusTeaching/multi_label_classifier/datasets/fashion-product-images-small


In [7]:
for each_item in downloaded_image_path.iterdir():
    print(each_item)

/Users/iputumahendrawijaya/BinusTeaching/multi_label_classifier/datasets/fashion-product-images-small/images
/Users/iputumahendrawijaya/BinusTeaching/multi_label_classifier/datasets/fashion-product-images-small/styles.csv
/Users/iputumahendrawijaya/BinusTeaching/multi_label_classifier/datasets/fashion-product-images-small/myntradataset


In [8]:
from typing import List, Set

def create_filtered_requirements(
    requirements_path: Path
) -> List[str]:
    """
    Remove conflicting packages from requirements.txt but still required in VertexAI training

    :param requirements_path: Path to requirements.txt
    :type requirements_path: Path

    :return: filtered_requirements:
    :rtype: List[str]
    """

    CONFLICTING_PACKAGES: Set = {
        "tensorflow",
        "pandas",
        "scikit-learn",
        "pillow",
        "numpy",
        "protobuf"
    }

    req_file: str = str(requirements_path)
    filtered_requirements: List[str] = []

    with open(req_file, "r") as req_file_handle:
        for each_line in req_file_handle:
            package_name: str = each_line.split("==")[0].strip().lower()
            if (
                package_name not in CONFLICTING_PACKAGES
                and
                not package_name.startswith("#")
            ):
                filtered_requirements.append(each_line.split("==")[0].strip())

    return filtered_requirements

In [9]:
import os

from google.cloud import aiplatform

from trainer.GCP.CredentialAccessor import CredentialAccessor as GcpCrAcc
from trainer.DaoCloudStorage.DaoCloudStorage import DaoCloudStorage as DaoGCS

if IS_LOCAL :
    print("Training Model in VertexAI Environment")

    # If local, train model in GCP Vertex AI
    gcp_cracc: GcpCrAcc = GcpCrAcc(
        credential_path = GCP_SERVICE_ACCOUNT_PATH,
        project_id = "disco-song-343611",
    )

    dao_gcs: DaoGCS = DaoGCS(
        credential_accessor = gcp_cracc,
    )

    # ⚠️This one line takes FOUR HOURS to complete
    # dao_gcs.upload_local_folder(
    #     gcs_bucket_name = "dsp_multi_label_classifier",
    #     local_folder_path = downloaded_image_path,
    #     gcs_folder_prefix = "datasets/fashion-product-images-small/"
    # )

    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(gcp_cracc.credential_path)

    aiplatform.init(
        project="disco-song-343611",
        location="asia-southeast1",
        staging_bucket="gs://dsp_multi_label_classifier_model/deploy/saved_models",
    )

    job: aiplatform.CustomTrainingJob = aiplatform.CustomTrainingJob(
        display_name="multi_label_classifier",
        script_path="trainer/task.py",
        container_uri="asia-docker.pkg.dev/vertex-ai/training/tf-gpu.2-16.py310:latest",
    )

    job.run(
        args=[
            "--epochs", "10",
            "--batch_size", "64",
            "--data_path", "/gcs/dsp_multi_label_classifier/datasets/fashion-product-images-small/"
        ],
        replica_count=1,
        machine_type="n1-standard-4",
        accelerator_type="NVIDIA_TESLA_T4",
        accelerator_count=1,
        sync=False
    )

elif IN_COLAB:
    print("Training Model in Colab Environment")

    result: object = subprocess.run(
        [
            sys.executable, "trainer/task.py",
            "--epochs", "10",
            "--batch_size", "64",
            "--data_path", str(downloaded_image_path)
        ],
        capture_output=True,
        text=True,
    )

    print(result)

Training Model in VertexAI Environment
